In [0]:
%sql select * from apexlife.silver.fact_visit ; 

diagnosis_code,hospital_id,patient_id,visit_id,admission_date,discharge_date,cost,first_name,last_name,gender,dob,patient_city,patient_first_last_name_masked,hospital_name,hospital_city,bed_count,diagnosis_desc
D003,H003,P001,V1001,2025-01-01,2025-01-05,12000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain
D005,H004,P003,V1005,2025-03-12,2025-03-18,22000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection
D005,H004,P003,V1006,2025-03-28,2025-03-31,16000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection
D003,H003,P001,V1002,2025-01-20,2025-01-22,9000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain
D002,H001,P005,V1008,2025-02-25,2025-03-02,20000,Vikram,Singh,M,1965-05-05,Chennai,deb97260b70ff54088f4b6cf6f75edf041b65eed5b26c608290dabc0e9efe0b6,Apollo Main Hospital,Chennai,850,Diabetes Type 2
D001,H002,P002,V1004,2025-04-01,2025-04-05,17000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension
D001,H002,P002,V1003,2025-02-10,2025-02-15,18000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension
D004,H005,P004,V1007,2025-01-15,2025-01-20,14000,Sneha,Rao,F,1988-09-19,Hyderabad,78a3e0a772632903fef2ffe51d6119d7df94f79dfb3295be92ecafe5029dc20f,Yashoda Hospital,Hyderabad,450,Asthma


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.table('apexlife.silver.fact_visit')

In [0]:
window = (
    Window
    .partitionBy(col('patient_id'))
    .orderBy(col('admission_date').asc())
)

In [0]:
df_with_prev = (
    df
    .withColumn('previous_discharge' , lag('admission_date').over(window))
    .withColumn('since_last_visit', date_diff('admission_date', 'previous_discharge'))
    .withColumn('is_readmission_30d' , when(col('since_last_visit') <= 30, 1).otherwise(0))
)

In [0]:
display(df_with_prev)

diagnosis_code,hospital_id,patient_id,visit_id,admission_date,discharge_date,cost,first_name,last_name,gender,dob,patient_city,patient_first_last_name_masked,hospital_name,hospital_city,bed_count,diagnosis_desc,previous_discharge,since_last_visit,is_readmission_30d
D003,H003,P001,V1001,2025-01-01,2025-01-05,12000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain,null,null,0
D003,H003,P001,V1002,2025-01-20,2025-01-22,9000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain,2025-01-01,19,1
D001,H002,P002,V1003,2025-02-10,2025-02-15,18000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension,null,null,0
D001,H002,P002,V1004,2025-04-01,2025-04-05,17000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension,2025-02-10,50,0
D005,H004,P003,V1005,2025-03-12,2025-03-18,22000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection,null,null,0
D005,H004,P003,V1006,2025-03-28,2025-03-31,16000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection,2025-03-12,16,1
D004,H005,P004,V1007,2025-01-15,2025-01-20,14000,Sneha,Rao,F,1988-09-19,Hyderabad,78a3e0a772632903fef2ffe51d6119d7df94f79dfb3295be92ecafe5029dc20f,Yashoda Hospital,Hyderabad,450,Asthma,null,null,0
D002,H001,P005,V1008,2025-02-25,2025-03-02,20000,Vikram,Singh,M,1965-05-05,Chennai,deb97260b70ff54088f4b6cf6f75edf041b65eed5b26c608290dabc0e9efe0b6,Apollo Main Hospital,Chennai,850,Diabetes Type 2,null,null,0


In [0]:
gold_df = (
    df_with_prev.
    groupBy([
        'hospital_id' ,
        'hospital_name' ,
        'diagnosis_desc'
    ])
    .agg(
        count('*').alias('total_visits'),
        sum('is_readmission_30d').alias('total_readmissions') ,
        round(sum('is_readmission_30d') / count('*'),2).alias('readmission_rate') ,
        sum("cost").alias("total_cost"),
        avg("cost").alias("avg_cost")
    )
    .withColumn("gold_load_timestamp", current_timestamp())
)

In [0]:
gold_df.write.mode('overwrite').saveAsTable('apexlife.gold.hospital_disease_kpi')

####  Which disease category causes maximum readmissions?

In [0]:
%sql select * from apexlife.gold.hospital_disease_kpi order by readmission_rate desc limit 1 ; 

hospital_id,hospital_name,diagnosis_desc,total_visits,total_readmissions,readmission_rate,total_cost,avg_cost,gold_load_timestamp
H004,Manipal Hospital,Kidney Infection,2,1,0.5,38000.0,19000.0,2026-06-07T15:16:54.215Z


#### Which disease category causes maximum readmissions?

In [0]:
%sql
select  
    diagnosis_desc ,
    sum(total_readmissions) as readmissions

from    
    apexlife.gold.hospital_disease_kpi
group by 
    diagnosis_desc
order by 
    readmissions desc

diagnosis_desc,readmissions
Chest Pain,1
Kidney Infection,1
Diabetes Type 2,0
Asthma,0
Hypertension,0
